In [107]:
# =============================================================
# CELL 1 — IMPORTS & SETUP
# Install and import all required libraries
# =============================================================

import sys
print(f"Python: {sys.version}")

!pip install pybaseball pandas

from pybaseball import statcast
print("pybaseball imported successfully")

Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
pybaseball imported successfully


In [108]:
# =============================================================
# CELL 2 — SESSION LOADER
# Load pre-classified checkpoint for repeat sessions
# Skip on first run — use cells 3-9 instead
# =============================================================
import pandas as pd
import os

save_path = r'E:\Projects\Data\ABS\python'

# Zone boundaries needed by analysis cells
HZ_MIN = -0.8333
HZ_MAX  =  0.8333

# Baseball tier thresholds in feet
BASEBALL_D = 0.24167
T1 = 0.00833
T2 = 0.06042
T3 = 0.12083
T4 = 0.24167
T5 = 0.48333

called_clean = pd.read_csv(
    os.path.join(save_path, 'called_clean_classified.csv')
)
print(f"Loaded: {len(called_clean):,} rows, "
      f"{len(called_clean.columns)} columns")
print(f"Years: {sorted(called_clean['game_year'].unique())}")

Loaded: 1,483,186 rows, 123 columns
Years: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [109]:
# =============================================================
# CELL 3 — PULL & SAVE SEASONS
# Pull 2022-2025 umpire decision pitches from Baseball Savant
# Includes: called_strike, ball, blocked_ball
# blocked_ball recoded as ball — both are umpire called balls
# =============================================================
import pandas as pd
import os
from pybaseball import statcast

save_path = r'E:\Projects\Data\ABS\python'
seasons = {
    '2022': ('2022-04-07', '2022-10-05'),
    '2023': ('2023-03-30', '2023-10-01'),
    '2024': ('2024-03-20', '2024-09-29'),
    '2025': ('2025-03-27', '2025-09-28')
}

for year, (start, end) in seasons.items():
    filepath = os.path.join(save_path, f'called_pitches_{year}.csv')

    if os.path.exists(filepath):
        df_check = pd.read_csv(filepath, usecols=['description'])
        if len(df_check) > 361450:
            print(f"{year}: already saved with blocked_ball, skipping")
            continue
        else:
            print(f"{year}: needs re-pull to add blocked_ball...")

    print(f"Pulling {year} data...")
    df = statcast(start_dt=start, end_dt=end)
    called = df[df['description'].isin([
        'called_strike',
        'ball',
        'blocked_ball'
    ])].copy()

    blocked_count = (called['description'] == 'blocked_ball').sum()
    called['description'] = called['description'].replace('blocked_ball', 'ball')

    called.to_csv(filepath, index=False)
    print(f"  Saved {len(called):,} umpire decision pitches")
    print(f"  Recoded {blocked_count:,} blocked_ball → ball")

print("\nAll seasons ready.")

2022: already saved with blocked_ball, skipping
2023: already saved with blocked_ball, skipping
2024: already saved with blocked_ball, skipping
2025: already saved with blocked_ball, skipping

All seasons ready.


In [110]:
# =============================================================
# CELL 4 — LOAD ALL SEASONS
# Combine four season CSVs into one dataframe
# =============================================================
called = pd.concat([
    pd.read_csv(os.path.join(save_path, 'called_pitches_2022.csv')),
    pd.read_csv(os.path.join(save_path, 'called_pitches_2023.csv')),
    pd.read_csv(os.path.join(save_path, 'called_pitches_2024.csv')),
    pd.read_csv(os.path.join(save_path, 'called_pitches_2025.csv'))
], ignore_index=True)

print(f"Total called pitches loaded: {len(called):,}")
print(f"Columns: {len(called.columns)}")
print(f"\nYears present: {sorted(called['game_year'].unique())}")
print(f"\nDescription breakdown:")
print(called['description'].value_counts())

Total called pitches loaded: 1,486,963
Columns: 118

Years present: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Description breakdown:
description
ball             1018263
called_strike     468700
Name: count, dtype: int64


In [111]:
# =============================================================
# CELL 5 — MISS CLASSIFICATION
# Flag each called pitch as correct, missed strike, or missed ball
# Uses ball-radius adjusted zone boundaries:
#   Horizontal: HZ_MIN/HZ_MAX already include ball radius
#   Vertical: sz_top + BALL_RADIUS, sz_bot - BALL_RADIUS
# =============================================================

called_clean = called.dropna(
    subset=['plate_x', 'plate_z', 'sz_top', 'sz_bot']
).copy()

print(f"Rows with full location data: {len(called_clean):,}")
print(f"Rows dropped (missing location): {len(called) - len(called_clean):,}")

HZ_MIN      = -0.8333
HZ_MAX       =  0.8333
BALL_RADIUS  =  0.1208  # feet — 1.45 inches

def classify_pitch(row):
    sz_top_adj = row['sz_top'] + BALL_RADIUS
    sz_bot_adj = row['sz_bot'] - BALL_RADIUS

    in_zone = (
        HZ_MIN <= row['plate_x'] <= HZ_MAX and
        sz_bot_adj <= row['plate_z'] <= sz_top_adj
    )
    if row['description'] == 'called_strike' and not in_zone:
        return 'missed_call_strike'
    elif row['description'] == 'ball' and in_zone:
        return 'missed_call_ball'
    else:
        return 'correct'

print("\nClassifying pitches with ball-radius adjusted zone...")
called_clean['call_result'] = called_clean.apply(classify_pitch, axis=1)

print("\nCall classification breakdown:")
print(called_clean['call_result'].value_counts())
print(f"\nOverall miss rate: "
      f"{(called_clean['call_result'] != 'correct').mean():.1%}")

Rows with full location data: 1,483,186
Rows dropped (missing location): 3,777

Classifying pitches with ball-radius adjusted zone...

Call classification breakdown:
call_result
correct               1374498
missed_call_strike      55945
missed_call_ball        52743
Name: count, dtype: int64

Overall miss rate: 7.3%


In [112]:
# =============================================================
# CELL 6 — ZONE DISTANCE CALCULATION
# miss_distance  = how far OUTSIDE the zone (for missed strikes)
# inside_distance = how far INSIDE the zone (for missed balls)
# Both use ball-radius adjusted boundaries
# =============================================================

BALL_RADIUS = 0.1208  # feet — 1.45 inches

def zone_distances(row):
    sz_top_adj = row['sz_top'] + BALL_RADIUS
    sz_bot_adj = row['sz_bot'] - BALL_RADIUS

    # --- OUTSIDE distance (for missed strikes) ---
    if row['plate_x'] < HZ_MIN:
        h_out = HZ_MIN - row['plate_x']
    elif row['plate_x'] > HZ_MAX:
        h_out = row['plate_x'] - HZ_MAX
    else:
        h_out = 0

    if row['plate_z'] < sz_bot_adj:
        v_out = sz_bot_adj - row['plate_z']
    elif row['plate_z'] > sz_top_adj:
        v_out = row['plate_z'] - sz_top_adj
    else:
        v_out = 0

    miss_dist = max(h_out, v_out)

    # --- INSIDE distance (for missed balls) ---
    # How comfortably inside the zone is the pitch center?
    # Min of distance to each edge — smallest margin = most inside metric
    if (HZ_MIN <= row['plate_x'] <= HZ_MAX and
        sz_bot_adj <= row['plate_z'] <= sz_top_adj):
        h_in = min(row['plate_x'] - HZ_MIN, HZ_MAX - row['plate_x'])
        v_in = min(row['plate_z'] - sz_bot_adj, sz_top_adj - row['plate_z'])
        inside_dist = min(h_in, v_in)
    else:
        inside_dist = 0

    return pd.Series([miss_dist, inside_dist])

print("Calculating zone distances...")
called_clean = called_clean.copy()
called_clean[['miss_distance', 'inside_distance']] = called_clean.apply(
    zone_distances, axis=1
)

print(f"Done.")
print(f"\nAvg miss_distance on missed strikes: "
      f"{called_clean[called_clean['call_result']=='missed_call_strike']['miss_distance'].mean()*12:.3f} inches")
print(f"Avg inside_distance on missed balls: "
      f"{called_clean[called_clean['call_result']=='missed_call_ball']['inside_distance'].mean()*12:.3f} inches")
print(f"\nMissed balls with inside_distance > 0: "
      f"{(called_clean[called_clean['call_result']=='missed_call_ball']['inside_distance'] > 0).sum():,}")
print(f"Missed balls with inside_distance = 0: "
      f"{(called_clean[called_clean['call_result']=='missed_call_ball']['inside_distance'] == 0).sum():,}")

Calculating zone distances...
Done.

Avg miss_distance on missed strikes: 1.170 inches
Avg inside_distance on missed balls: 1.191 inches

Missed balls with inside_distance > 0: 52,743
Missed balls with inside_distance = 0: 0


In [113]:
# =============================================================
# CELL 7 — FULL PITCH OUTCOME BREAKDOWN BY SEASON
# Exact counts for every pitch outcome 2022-2025
# Shows readers which pitches are ABS-relevant vs not
# =============================================================

print("=== FULL PITCH OUTCOME BREAKDOWN BY SEASON ===\n")

seasons_data = {
    '2022': {
        'ball': 235203, 'called_strike': 115882,
        'blocked_ball': 16060,
        'foul': 127216, 'hit_into_play': 124275,
        'swinging_strike': 74987,
        'swinging_strike_blocked': 4474,
        'foul_tip': 7008,
        'hit_by_pitch': 2046,
        'automatic_ball': 1670, 'automatic_strike': 0,
        'foul_bunt': 1111, 'missed_bunt': 214,
        'pitchout': 40, 'bunt_foul_tip': 24
    },
    '2023': {
        'ball': 240670, 'called_strike': 117936,
        'blocked_ball': 15917,
        'foul': 127939, 'hit_into_play': 124246,
        'swinging_strike': 76298,
        'swinging_strike_blocked': 4322,
        'foul_tip': 7256,
        'hit_by_pitch': 2112,
        'automatic_ball': 2437, 'automatic_strike': 302,
        'foul_bunt': 1002, 'missed_bunt': 185,
        'pitchout': 46, 'bunt_foul_tip': 16
    },
    '2024': {
        'ball': 242055, 'called_strike': 119395,
        'blocked_ball': 15250,
        'foul': 131425, 'hit_into_play': 128385,
        'swinging_strike': 77801,
        'swinging_strike_blocked': 4030,
        'foul_tip': 7507,
        'hit_by_pitch': 2101,
        'automatic_ball': 2276, 'automatic_strike': 147,
        'foul_bunt': 1255, 'missed_bunt': 206,
        'pitchout': 55, 'bunt_foul_tip': 16
    },
    '2025': {
        'ball': 237986, 'called_strike': 115487,
        'blocked_ball': 15122,
        'foul': 127268, 'hit_into_play': 124809,
        'swinging_strike': 74033,
        'swinging_strike_blocked': 3990,
        'foul_tip': 7297,
        'hit_by_pitch': 1927,
        'automatic_ball': 2342, 'automatic_strike': 96,
        'foul_bunt': 1245, 'missed_bunt': 208,
        'pitchout': 54, 'bunt_foul_tip': 33
    }
}

for year, s in seasons_data.items():
    total = sum(s.values())

    # ABS relevant — umpire judgment calls only
    ump_decisions = s['ball'] + s['called_strike'] + s['blocked_ball']

    # Batter decisions — swings of all kinds
    batter_swing  = (s['swinging_strike'] + s['swinging_strike_blocked'] +
                     s['foul_tip'] + s['foul'] + s['foul_bunt'] +
                     s['missed_bunt'] + s['bunt_foul_tip'] +
                     s['hit_into_play'])

    # Automatic — pitch clock, not umpire judgment
    automatic     = s['automatic_ball'] + s['automatic_strike']

    # Other — HBP, pitchout
    other         = s['hit_by_pitch'] + s['pitchout']

    print(f"{year} — {total:,} total pitches:")
    print(f"  UMPIRE DECISIONS (ABS relevant):   "
          f"{ump_decisions:>8,}  ({ump_decisions/total:>5.1%})")
    print(f"    Called balls:                    "
          f"{s['ball']:>8,}  ({s['ball']/total:>5.1%})")
    print(f"    Called strikes:                  "
          f"{s['called_strike']:>8,}  ({s['called_strike']/total:>5.1%})")
    print(f"    Blocked balls (called):          "
          f"{s['blocked_ball']:>8,}  ({s['blocked_ball']/total:>5.1%})")
    print(f"  BATTER DECISIONS (ABS irrelevant): "
          f"{batter_swing:>8,}  ({batter_swing/total:>5.1%})")
    print(f"    Hit into play:                   "
          f"{s['hit_into_play']:>8,}  ({s['hit_into_play']/total:>5.1%})")
    print(f"    Foul balls:                      "
          f"{s['foul']:>8,}  ({s['foul']/total:>5.1%})")
    print(f"    Swinging strikes:                "
          f"{s['swinging_strike']:>8,}  ({s['swinging_strike']/total:>5.1%})")
    print(f"    Other swings (blocked/tip/bunt): "
          f"{s['swinging_strike_blocked']+s['foul_tip']+s['foul_bunt']+s['missed_bunt']+s['bunt_foul_tip']:>8,}  "
          f"({(s['swinging_strike_blocked']+s['foul_tip']+s['foul_bunt']+s['missed_bunt']+s['bunt_foul_tip'])/total:>5.1%})")
    print(f"  AUTOMATIC (pitch clock):           "
          f"{automatic:>8,}  ({automatic/total:>5.1%})")
    print(f"  OTHER (HBP, pitchout):             "
          f"{other:>8,}  ({other/total:>5.1%})")
    print()

=== FULL PITCH OUTCOME BREAKDOWN BY SEASON ===

2022 — 710,210 total pitches:
  UMPIRE DECISIONS (ABS relevant):    367,145  (51.7%)
    Called balls:                     235,203  (33.1%)
    Called strikes:                   115,882  (16.3%)
    Blocked balls (called):            16,060  ( 2.3%)
  BATTER DECISIONS (ABS irrelevant):  339,309  (47.8%)
    Hit into play:                    124,275  (17.5%)
    Foul balls:                       127,216  (17.9%)
    Swinging strikes:                  74,987  (10.6%)
    Other swings (blocked/tip/bunt):   12,831  ( 1.8%)
  AUTOMATIC (pitch clock):              1,670  ( 0.2%)
  OTHER (HBP, pitchout):                2,086  ( 0.3%)

2023 — 720,684 total pitches:
  UMPIRE DECISIONS (ABS relevant):    374,523  (52.0%)
    Called balls:                     240,670  (33.4%)
    Called strikes:                   117,936  (16.4%)
    Blocked balls (called):            15,917  ( 2.2%)
  BATTER DECISIONS (ABS irrelevant):  341,264  (47.4%)
    Hit int

In [114]:
# =============================================================
# CELL 8 — TIERED CLASSIFICATION
# Classify missed calls into six severity tiers
# Missed strikes use miss_distance (how far outside zone)
# Missed balls use inside_distance (how far inside zone)
# =============================================================

BASEBALL_D = 0.24167  # 2.9 inches
T1 = 0.00833   # 0.1 inches — broadcast threshold
T2 = 0.06042   # 0.725 inches — 1/4 baseball
T3 = 0.12083   # 1.45 inches — 1/2 baseball
T4 = 0.24167   # 2.9 inches — 1 baseball
T5 = 0.48333   # 5.8 inches — 2 baseballs

def classify_call_tiered(row):
    if row['call_result'] == 'missed_call_strike':
        dist = row['miss_distance']
        direction = 'missed_strike'
    elif row['call_result'] == 'missed_call_ball':
        dist = row['inside_distance']
        direction = 'missed_ball'
    else:
        return 'correct'

    if dist < T1:
        tier = 'T1_broadcast'
    elif dist < T2:
        tier = 'T2_quarter_ball'
    elif dist < T3:
        tier = 'T3_half_ball'
    elif dist < T4:
        tier = 'T4_one_ball'
    elif dist < T5:
        tier = 'T5_two_balls'
    else:
        tier = 'T6_egregious'

    return f"{direction}_{tier}"

print("Classifying with tiered thresholds...")
called_clean['call_result_tiered'] = called_clean.apply(
    classify_call_tiered, axis=1
)

# Summary
print("\n=== TIERED CLASSIFICATION BREAKDOWN ===")
print(f"\n{'Category':<45} {'Count':>8}  {'% of All':>8}  {'% of Missed':>12}")
print("-" * 78)

total = len(called_clean)
missed = called_clean[called_clean['call_result_tiered'] != 'correct']
total_missed = len(missed)

correct_count = len(called_clean[called_clean['call_result_tiered'] == 'correct'])
print(f"{'correct':<45} {correct_count:>8,}  {correct_count/total:>7.1%}")
print()

categories = [
    ('missed_strike_T1_broadcast',    '< 0.1\" (broadcast threshold)'),
    ('missed_strike_T2_quarter_ball', '0.1\" to ¼ baseball'),
    ('missed_strike_T3_half_ball',    '¼ to ½ baseball'),
    ('missed_strike_T4_one_ball',     '½ to 1 baseball'),
    ('missed_strike_T5_two_balls',    '1 to 2 baseballs'),
    ('missed_strike_T6_egregious',    '2+ baseballs'),
]

print("MISSED STRIKES (called ball, inside zone):")
strike_total = 0
for code, label in categories:
    n = len(called_clean[called_clean['call_result_tiered'] == code])
    strike_total += n
    print(f"  {label:<43} {n:>8,}  {n/total:>7.1%}  {n/total_missed:>11.1%}")
print(f"  {'SUBTOTAL':<43} {strike_total:>8,}  "
      f"{strike_total/total:>7.1%}  "
      f"{strike_total/total_missed:>11.1%}")

print()
ball_categories = [
    ('missed_ball_T1_broadcast',    '< 0.1\" (broadcast threshold)'),
    ('missed_ball_T2_quarter_ball', '0.1\" to ¼ baseball'),
    ('missed_ball_T3_half_ball',    '¼ to ½ baseball'),
    ('missed_ball_T4_one_ball',     '½ to 1 baseball'),
    ('missed_ball_T5_two_balls',    '1 to 2 baseballs'),
    ('missed_ball_T6_egregious',    '2+ baseballs'),
]

print("MISSED BALLS (called strike, outside zone):")
ball_total = 0
for code, label in ball_categories:
    n = len(called_clean[called_clean['call_result_tiered'] == code])
    ball_total += n
    print(f"  {label:<43} {n:>8,}  {n/total:>7.1%}  {n/total_missed:>11.1%}")
print(f"  {'SUBTOTAL':<43} {ball_total:>8,}  "
      f"{ball_total/total:>7.1%}  "
      f"{ball_total/total_missed:>11.1%}")

print()
print(f"{'TOTAL MISSED':<45} {total_missed:>8,}  "
      f"{total_missed/total:>7.1%}  {'100.0%':>11}")

Classifying with tiered thresholds...

=== TIERED CLASSIFICATION BREAKDOWN ===

Category                                         Count  % of All   % of Missed
------------------------------------------------------------------------------
correct                                       1,374,498    92.7%

MISSED STRIKES (called ball, inside zone):
  < 0.1" (broadcast threshold)                   3,552     0.2%         3.3%
  0.1" to ¼ baseball                            19,061     1.3%        17.5%
  ¼ to ½ baseball                               16,125     1.1%        14.8%
  ½ to 1 baseball                               13,806     0.9%        12.7%
  1 to 2 baseballs                               3,307     0.2%         3.0%
  2+ baseballs                                      94     0.0%         0.1%
  SUBTOTAL                                      55,945     3.8%        51.5%

MISSED BALLS (called strike, outside zone):
  < 0.1" (broadcast threshold)                   3,698     0.2%      

In [115]:
# =============================================================
# CELL 9 — BATTER HEIGHT JOIN
# Pull player heights from MLB Stats API
# Joins to called_clean via batter MLBAM ID
# =============================================================

import requests
import time

# Get all unique batter IDs from our dataset
batter_ids = called_clean['batter'].dropna().unique()
print(f"Unique batters to look up: {len(batter_ids):,}")

def get_player_height(mlbam_id):
    try:
        url = f"https://statsapi.mlb.com/api/v1/people/{int(mlbam_id)}"
        params = {"fields": "people,id,fullName,height"}
        r = requests.get(url, params=params, timeout=5)
        data = r.json()
        height_str = data['people'][0].get('height', None)
        return height_str
    except:
        return None

def parse_height_inches(height_str):
    if not height_str:
        return None
    try:
        # Format is "6' 7""
        parts = height_str.replace('"', '').split("'")
        feet = int(parts[0].strip())
        inches = int(parts[1].strip())
        return (feet * 12) + inches
    except:
        return None

# Pull height for all unique batters
print("Pulling heights from MLB Stats API...")
print("(this will take a few minutes due to rate limiting)\n")

height_lookup = {}
for i, batter_id in enumerate(batter_ids):
    height_str = get_player_height(batter_id)
    height_lookup[batter_id] = parse_height_inches(height_str)
    
    # Progress update every 100 players
    if (i + 1) % 100 == 0:
        print(f"  {i+1:,} of {len(batter_ids):,} completed...")
    
    # Rate limit — be respectful to the API
    time.sleep(0.1)

print(f"\nDone. Heights retrieved: "
      f"{sum(1 for v in height_lookup.values() if v is not None):,} "
      f"of {len(batter_ids):,}")

# Join to called_clean
called_clean['batter_height_in'] = called_clean['batter'].map(height_lookup)

print(f"\nHeight coverage in dataset: "
      f"{called_clean['batter_height_in'].notna().sum():,} rows "
      f"({called_clean['batter_height_in'].notna().mean():.1%})")
print(f"\nHeight distribution (inches):")
print(called_clean['batter_height_in'].describe())

Unique batters to look up: 1,279
Pulling heights from MLB Stats API...
(this will take a few minutes due to rate limiting)

  100 of 1,279 completed...
  200 of 1,279 completed...
  300 of 1,279 completed...
  400 of 1,279 completed...
  500 of 1,279 completed...
  600 of 1,279 completed...
  700 of 1,279 completed...
  800 of 1,279 completed...
  900 of 1,279 completed...
  1,000 of 1,279 completed...
  1,100 of 1,279 completed...
  1,200 of 1,279 completed...

Done. Heights retrieved: 1,279 of 1,279

Height coverage in dataset: 1,483,186 rows (100.0%)

Height distribution (inches):
count    1.483186e+06
mean     7.224520e+01
std      2.324995e+00
min      6.500000e+01
25%      7.100000e+01
50%      7.200000e+01
75%      7.400000e+01
max      8.000000e+01
Name: batter_height_in, dtype: float64


In [116]:
# =============================================================
# CELL 10 — CHAPTER 1: THE SCALE
# How many missed calls, per season, per game?
# Year over year trend, borderline vs egregious breakdown
# =============================================================

print("=== CHAPTER 1: THE SCALE OF THE PROBLEM ===\n")

# Overall miss rate by season
print("--- MISS RATE BY SEASON ---\n")
print(f"{'Season':<8} {'Decisions':>10} {'Missed':>8} {'Miss%':>7} "
      f"{'Games':>6} {'Per Game':>9} {'Correct%':>9}")
print("-" * 65)

total_missed_all = 0
total_decisions_all = 0

for year in [2022, 2023, 2024, 2025]:
    yr = called_clean[called_clean['game_year'] == year]
    total = len(yr)
    missed = len(yr[yr['call_result'] != 'correct'])
    correct = total - missed
    games = yr['game_pk'].nunique()
    total_missed_all += missed
    total_decisions_all += total

    print(f"{year:<8} {total:>10,} {missed:>8,} {missed/total:>7.1%} "
          f"{games:>6,} {missed/games:>9.1f} {correct/total:>9.1%}")

print("-" * 65)
print(f"{'TOTAL':<8} {total_decisions_all:>10,} {total_missed_all:>8,} "
      f"{total_missed_all/total_decisions_all:>7.1%}")

# Borderline vs egregious by season
print(f"\n\n--- MISSED CALL SEVERITY BY SEASON ---\n")
print(f"{'Season':<8} {'Total Miss':>11} {'Broadcast':>10} "
      f"{'¼ Ball':>8} {'½ Ball':>8} "
      f"{'1 Ball':>8} {'2 Ball':>8} {'Egregious':>10}")
print("-" * 80)

for year in [2022, 2023, 2024, 2025]:
    yr = called_clean[called_clean['game_year'] == year]
    missed = yr[yr['call_result'] != 'correct']
    total_miss = len(missed)

    t1 = len(yr[yr['call_result_tiered'].str.contains('T1', na=False)])
    t2 = len(yr[yr['call_result_tiered'].str.contains('T2', na=False)])
    t3 = len(yr[yr['call_result_tiered'].str.contains('T3', na=False)])
    t4 = len(yr[yr['call_result_tiered'].str.contains('T4', na=False)])
    t5 = len(yr[yr['call_result_tiered'].str.contains('T5', na=False)])
    t6 = len(yr[yr['call_result_tiered'].str.contains('T6', na=False)])

    print(f"{year:<8} {total_miss:>11,} {t1:>10,} {t2:>8,} "
          f"{t3:>8,} {t4:>8,} {t5:>8,} {t6:>10,}")

# Missed strikes vs missed balls ratio by season
print(f"\n\n--- MISSED STRIKES vs MISSED BALLS BY SEASON ---\n")
print(f"{'Season':<8} {'Miss Strike':>12} {'Miss Ball':>10} "
      f"{'Ratio':>8} {'Strike%':>9} {'Ball%':>8}")
print("-" * 60)

for year in [2022, 2023, 2024, 2025]:
    yr = called_clean[called_clean['game_year'] == year]
    ms = len(yr[yr['call_result'] == 'missed_call_strike'])
    mb = len(yr[yr['call_result'] == 'missed_call_ball'])
    total_miss = ms + mb
    ratio = ms / mb if mb > 0 else 0

    print(f"{year:<8} {ms:>12,} {mb:>10,} "
          f"{ratio:>7.1f}x {ms/total_miss:>8.1%} {mb/total_miss:>8.1%}")

print(f"\n\n--- HEADLINE NUMBERS FOR THE HOOK ---\n")
for year in [2022, 2023, 2024, 2025]:
    yr = called_clean[called_clean['game_year'] == year]
    missed = len(yr[yr['call_result'] != 'correct'])
    games = yr['game_pk'].nunique()
    print(f"{year}: {missed:,} missed calls — "
          f"{missed/games:.1f} per game across {games:,} games")

=== CHAPTER 1: THE SCALE OF THE PROBLEM ===

--- MISS RATE BY SEASON ---

Season    Decisions   Missed   Miss%  Games  Per Game  Correct%
-----------------------------------------------------------------
2022        366,941   27,543    7.5%  2,429      11.3     92.5%
2023        374,385   26,832    7.2%  2,430      11.0     92.8%
2024        373,345   27,833    7.5%  2,471      11.3     92.5%
2025        368,515   26,480    7.2%  2,428      10.9     92.8%
-----------------------------------------------------------------
TOTAL     1,483,186  108,688    7.3%


--- MISSED CALL SEVERITY BY SEASON ---

Season    Total Miss  Broadcast   ¼ Ball   ½ Ball   1 Ball   2 Ball  Egregious
--------------------------------------------------------------------------------
2022          27,543      1,771    9,493    7,449    6,815    1,960         55
2023          26,832      1,762    9,445    7,207    6,373    1,993         52
2024          27,833      1,869    9,727    7,530    6,646    2,007         5

In [117]:
# =============================================================
# CELL 11 — COUNT SITUATION ANALYSIS (Chapter 2)
# Miss rates by ball-strike count
# Proves systematic bias — not random human error
# =============================================================

print("=== CHAPTER 2: COUNTING THE COUNT ===\n")

# Create a readable count label
called_clean['count'] = (called_clean['balls'].astype(int).astype(str) + 
                         '-' + 
                         called_clean['strikes'].astype(int).astype(str))

# Miss rate by count
count_summary = (called_clean
    .groupby('count')
    .agg(
        total        = ('call_result', 'count'),
        missed       = ('call_result', lambda x: (x != 'correct').sum()),
        missed_strike= ('call_result', lambda x: (x == 'missed_call_strike').sum()),
        missed_ball  = ('call_result', lambda x: (x == 'missed_call_ball').sum()),
    )
    .assign(
        miss_rate        = lambda x: x['missed']        / x['total'],
        missed_str_rate  = lambda x: x['missed_strike'] / x['total'],
        missed_ball_rate = lambda x: x['missed_ball']   / x['total'],
    )
    .sort_values('miss_rate', ascending=False)
)

print("=== MISS RATE BY COUNT ===\n")
print(f"{'Count':<8} {'Total':>8} {'Missed':>8} {'Miss%':>7} "
      f"{'Miss Strike%':>13} {'Miss Ball%':>11}")
print("-" * 60)
for count, row in count_summary.iterrows():
    print(f"{count:<8} {row['total']:>8,} {row['missed']:>8,} "
          f"{row['miss_rate']:>7.1%} {row['missed_str_rate']:>13.1%} "
          f"{row['missed_ball_rate']:>11.1%}")

=== CHAPTER 2: COUNTING THE COUNT ===

=== MISS RATE BY COUNT ===

Count       Total   Missed   Miss%  Miss Strike%  Miss Ball%
------------------------------------------------------------
2-0      53,638.0  5,020.0    9.4%          6.1%        3.2%
3-1      27,697.0  2,449.0    8.8%          5.0%        3.8%
1-0      158,198.0 13,694.0    8.7%          5.2%        3.4%
3-0      25,814.0  2,179.0    8.4%          5.6%        2.9%
0-0      500,040.0 42,006.0    8.4%          4.7%        3.7%
2-1      61,887.0  5,162.0    8.3%          4.5%        3.9%
1-1      130,364.0  9,476.0    7.3%          3.4%        3.9%
3-2      41,805.0  2,950.0    7.1%          3.5%        3.6%
0-1      188,149.0 12,878.0    6.8%          2.6%        4.3%
2-2      83,337.0  4,551.0    5.5%          2.3%        3.2%
1-2      117,489.0  4,934.0    4.2%          1.5%        2.7%
0-2      94,768.0  3,389.0    3.6%          1.0%        2.5%


In [118]:
# =============================================================
# CELL 12 — CHAPTER 3: THE PITCH
# Miss rates by pitch type
# Are certain pitches miscalled more than others?
# =============================================================

print("=== CHAPTER 3: WHICH PITCH ===\n")

# Display names for pitch type codes
pitch_names = {
    'FF': '4-Seam Fastball',
    'SI': 'Sinker',
    'SL': 'Slider',
    'CH': 'Changeup',
    'FC': 'Cutter',
    'CU': 'Curveball',
    'ST': 'Sweeper',
    'FS': 'Splitter',
    'KC': 'Knuckle Curve',
    'SV': 'Slurve',
    'FA': 'Fastball',
    'EP': 'Eephus',
    'KN': 'Knuckleball',
    'FO': 'Forkball',
    'CS': 'Slow Curve',
    'SC': 'Screwball',
    'UN': 'Unknown'
}

# Miss rate by pitch type
pitch_summary = (called_clean
    .groupby('pitch_type')
    .agg(
        total         = ('call_result', 'count'),
        missed        = ('call_result', lambda x: (x != 'correct').sum()),
        missed_strike = ('call_result', lambda x: (x == 'missed_call_strike').sum()),
        missed_ball   = ('call_result', lambda x: (x == 'missed_call_ball').sum()),
    )
    .assign(
        miss_rate        = lambda x: x['missed']         / x['total'],
        missed_str_rate  = lambda x: x['missed_strike']  / x['total'],
        missed_ball_rate = lambda x: x['missed_ball']    / x['total'],
        display_name     = lambda x: x.index.map(pitch_names)
    )
    .sort_values('miss_rate', ascending=False)
)

print("=== MISS RATE BY PITCH TYPE ===\n")
print(f"{'Pitch Type':<22} {'Code':<6} {'Total':>8} {'Missed':>8} "
      f"{'Miss%':>7} {'MissStr%':>9} {'MissBall%':>10}")
print("-" * 75)

for code, row in pitch_summary.iterrows():
    # Skip very rare pitch types (under 500 pitches — not statistically meaningful)
    if row['total'] < 500:
        continue
    name = pitch_names.get(code, code)
    print(f"{name:<22} {code:<6} {row['total']:>8,} {row['missed']:>8,} "
          f"{row['miss_rate']:>7.1%} {row['missed_str_rate']:>9.1%} "
          f"{row['missed_ball_rate']:>10.1%}")

print(f"\nNote: Pitch types with fewer than 500 called pitches excluded")
print(f"      (Slow Curve, Screwball, Unknown)")

=== CHAPTER 3: WHICH PITCH ===

=== MISS RATE BY PITCH TYPE ===

Pitch Type             Code      Total   Missed   Miss%  MissStr%  MissBall%
---------------------------------------------------------------------------
Eephus                 EP        1,417      192   13.5%     10.2%       3.4%
Sinker                 SI      239,181   21,387    8.9%      4.6%       4.3%
Fastball               FA        2,052      176    8.6%      6.4%       2.1%
4-Seam Fastball        FF      471,473   39,025    8.3%      4.0%       4.3%
Cutter                 FC      112,269    9,124    8.1%      4.3%       3.8%
Slurve                 SV        7,166      457    6.4%      3.8%       2.6%
Slider                 SL      225,825   14,277    6.3%      3.5%       2.8%
Curveball              CU      112,112    6,972    6.2%      3.3%       2.9%
Sweeper                ST       94,382    5,832    6.2%      3.4%       2.8%
Knuckle Curve          KC       31,113    1,867    6.0%      3.1%       2.9%
Knuckleball 

In [124]:
# =============================================================
# CELL 13 — CHAPTER 4: DOES HEIGHT MATTER?
# Miss rates by batter height bucket
# Do taller/shorter batters experience different miss rates?
# Where do the misses happen — top or bottom of zone?
# =============================================================

# Create height buckets
def height_bucket(inches):
    if inches <= 68:
        return '1_short'        # 5\'8\" and under
    elif inches <= 71:
        return '2_below_avg'    # 5\'9\" to 5\'11\"
    elif inches <= 73:
        return '3_average'      # 6\'0\" to 6\'1\"
    elif inches <= 75:
        return '4_above_avg'    # 6\'2\" to 6\'3\"
    else:
        return '5_tall'         # 6\'4\" and over

height_labels = {
    '1_short':      '5\'8\" and under',
    '2_below_avg':  '5\'9\" to 5\'11\"',
    '3_average':    '6\'0\" to 6\'1\"',
    '4_above_avg':  '6\'2\" to 6\'3\"',
    '5_tall':       '6\'4\" and over'
}

called_clean['height_bucket'] = called_clean['batter_height_in'].apply(
    height_bucket
)

print("=== CHAPTER 4: DOES HEIGHT MATTER? ===\n")

# --- MISS LOCATION BY HEIGHT (Top vs Bottom of Zone) ---
print(f"\n\n--- MISS LOCATION BY HEIGHT (Top vs Bottom of Zone) ---\n")
print(f"{'Height':<20} {'Miss Top%':>10} {'Miss Bot%':>10} "
      f"{'Str Top%':>10} {'Str Bot%':>10} "
      f"{'Ball Top%':>10} {'Ball Bot%':>10}")
print("-" * 85)

for bucket in sorted(called_clean['height_bucket'].dropna().unique()):
    yr = called_clean[called_clean['height_bucket'] == bucket].reset_index(drop=True)
    missed = yr[yr['call_result'] != 'correct'].reset_index(drop=True)
    ms = yr[yr['call_result'] == 'missed_call_strike'].reset_index(drop=True)
    mb = yr[yr['call_result'] == 'missed_call_ball'].reset_index(drop=True)

    zone_mid_missed = (missed['sz_top'] + missed['sz_bot']) / 2
    zone_mid_ms     = (ms['sz_top'] + ms['sz_bot']) / 2
    zone_mid_mb     = (mb['sz_top'] + mb['sz_bot']) / 2

    miss_top = len(missed[missed['plate_z'] > zone_mid_missed])
    miss_bot = len(missed[missed['plate_z'] <= zone_mid_missed])
    total_miss = len(missed)

    ms_top = len(ms[ms['plate_z'] > zone_mid_ms])
    ms_bot = len(ms) - ms_top

    mb_top = len(mb[mb['plate_z'] > zone_mid_mb])
    mb_bot = len(mb) - mb_top

    label = height_labels.get(bucket, bucket)
    print(f"{label:<20} "
          f"{miss_top/total_miss:>10.1%} {miss_bot/total_miss:>10.1%} "
          f"{ms_top/len(ms) if len(ms)>0 else 0:>10.1%} "
          f"{ms_bot/len(ms) if len(ms)>0 else 0:>10.1%} "
          f"{mb_top/len(mb) if len(mb)>0 else 0:>10.1%} "
          f"{mb_bot/len(mb) if len(mb)>0 else 0:>10.1%}")

# Extreme height comparison
print(f"\n\n--- EXTREME HEIGHT COMPARISON ---\n")

short = called_clean[called_clean['batter_height_in'] <= 67].reset_index(drop=True)
tall  = called_clean[called_clean['batter_height_in'] >= 76].reset_index(drop=True)

for label, group in [('Short (5\'7\" and under)', short),
                      ('Tall (6\'4\" and over)',   tall)]:
    total  = len(group)
    ms_df  = group[group['call_result'] == 'missed_call_strike'].reset_index(drop=True)
    mb_df  = group[group['call_result'] == 'missed_call_ball'].reset_index(drop=True)
    ms     = len(ms_df)
    mb     = len(mb_df)
    missed = ms + mb

    zone_mid_ms = (ms_df['sz_top'] + ms_df['sz_bot']) / 2
    zone_mid_mb = (mb_df['sz_top'] + mb_df['sz_bot']) / 2

    ms_top = len(ms_df[ms_df['plate_z'] > zone_mid_ms])
    ms_bot = ms - ms_top
    mb_top = len(mb_df[mb_df['plate_z'] > zone_mid_mb])
    mb_bot = mb - mb_top

    print(f"{label}:")
    print(f"  Total decisions:   {total:,}")
    print(f"  Overall miss rate: {missed/total:.1%}")
    print(f"  Missed strikes:    {ms:,} ({ms/total:.1%}) — "
          f"top: {ms_top/ms:.1%}  bot: {ms_bot/ms:.1%}")
    print(f"  Missed balls:      {mb:,} ({mb/total:.1%}) — "
          f"top: {mb_top/mb:.1%}  bot: {mb_bot/mb:.1%}")
    print()

=== CHAPTER 4: DOES HEIGHT MATTER? ===



--- MISS LOCATION BY HEIGHT (Top vs Bottom of Zone) ---

Height                Miss Top%  Miss Bot%   Str Top%   Str Bot%  Ball Top%  Ball Bot%
-------------------------------------------------------------------------------------
5'8" and under            52.0%      48.0%      61.2%      38.8%      42.0%      58.0%
5'9" to 5'11"             47.6%      52.4%      51.4%      48.6%      43.5%      56.5%
6'0" to 6'1"              45.6%      54.4%      39.6%      60.4%      52.3%      47.7%
6'2" to 6'3"              46.0%      54.0%      36.0%      64.0%      56.3%      43.7%
6'4" and over             46.0%      54.0%      32.4%      67.6%      59.2%      40.8%


--- EXTREME HEIGHT COMPARISON ---

Short (5'7" and under):
  Total decisions:   22,764
  Overall miss rate: 7.7%
  Missed strikes:    920 (4.0%) — top: 65.2%  bot: 34.8%
  Missed balls:      840 (3.7%) — top: 42.9%  bot: 57.1%

Tall (6'4" and over):
  Total decisions:   134,188
  Overall mi

In [125]:
# =============================================================
# CELL LAST — SAVE CHECKPOINT
# Save called_clean with all new columns to disk
# Re-run any time new columns are added
# =============================================================

called_clean.to_csv(
    os.path.join(save_path, 'called_clean_classified.csv'),
    index=False
)
print(f"Saved: {len(called_clean):,} rows, "
      f"{len(called_clean.columns)} columns")

Saved: 1,483,186 rows, 125 columns
